In [11]:
import os
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO


# paths
VIDEO_PATH = r"N:\Final Preparation\FULL_DATASET\new\testing\positive\theft_082.mp4"
MODEL_PATH = r"N:\Final Preparation\best_theft_lstm_model_v6.pth"
YOLO_MODEL = "yolov8m-pose.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_FPS                 = 24
CONF_THRESHOLD             = 0.3
IOU_THRESHOLD              = 0.5
SPATIAL_MATCHING_THRESHOLD = 0.25
MAX_FRAMES_TO_REMEMBER     = 100
MERGE_TEMPORAL_GAP         = 50

WINDOW_SIZE      = 30
STRIDE           = 10
MIN_SEQUENCE_LEN = 25
INPUT_SIZE       = 17 * 3   # 51

HIDDEN_SIZE = 48
NUM_LAYERS  = 1
DROPOUT     = 0.6

PRED_THRESHOLD_OVERRIDE = None
VERDICT_MODE            = "topk_avg"
TOP_K_WINDOWS           = 3
SHOW_PER_SEQUENCE_PROBS = True


class TheftDetectionLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.lstm    = nn.LSTM(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
                               num_layers=NUM_LAYERS, batch_first=True, dropout=0.0)
        self.bn      = nn.BatchNorm1d(HIDDEN_SIZE)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        return self.fc(out)


class PersonTracker:

    def __init__(self):
        self.sequences      = defaultdict(list)
        self.person_history = {}
        self.next_id        = 0

    def process_video(self, yolo_model, video_path):
        cap          = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        source_fps   = cap.get(cv2.CAP_PROP_FPS)
        fps_ratio    = source_fps / TARGET_FPS if TARGET_FPS > 0 else 1.0

        print(f"video: {total_frames} frames | {frame_w}x{frame_h} | {source_fps:.1f}fps -> {TARGET_FPS}fps")

        frame_idx        = 0
        frames_processed = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            curr_bucket = int(frame_idx / fps_ratio)
            prev_bucket = int((frame_idx - 1) / fps_ratio) if frame_idx > 0 else -1

            if curr_bucket > prev_bucket:
                results = yolo_model(frame, conf=CONF_THRESHOLD,
                                     iou=IOU_THRESHOLD, verbose=False)
                r = results[0]

                if r.boxes is not None and r.keypoints is not None and len(r.boxes) > 0:
                    boxes    = r.boxes.xyxy.cpu().numpy()
                    kpts_all = r.keypoints.data.cpu().numpy()

                    for box, kpts in zip(boxes, kpts_all):
                        pid = self._resolve_id(box, frames_processed)
                        self._update_history(pid, box, frames_processed)
                        self.sequences[pid].append(
                            normalize_keypoints(kpts, frame_w, frame_h)
                        )

                frames_processed += 1

            frame_idx += 1
            if frame_idx % 100 == 0:
                pct = frame_idx / total_frames * 100 if total_frames > 0 else 0
                print(f"  tracking {frame_idx}/{total_frames} ({pct:.0f}%)", end="\r")

        cap.release()
        print(f"  tracking done — {frames_processed} frames | {len(self.sequences)} track(s)    ")
        return dict(self.sequences)

    def _resolve_id(self, box, frame_idx):
        best_match, best_score = None, 0.0

        for pid, history in self.person_history.items():
            frames_since = frame_idx - history["last_seen"]
            if frames_since > MAX_FRAMES_TO_REMEMBER:
                continue
            if   frames_since > 30: thresh = 0.15
            elif frames_since > 10: thresh = 0.20
            else:                   thresh = SPATIAL_MATCHING_THRESHOLD

            recent  = history["bboxes"][-10:]
            ious    = [calculate_iou(box.tolist(), b) for b in recent]
            avg_iou = float(np.mean(ious)) if ious else 0.0

            if avg_iou > thresh and avg_iou > best_score:
                best_score = avg_iou
                best_match = pid

        if best_match is not None:
            return best_match

        new_id = self.next_id
        self.person_history[new_id] = {"bboxes": [], "last_seen": frame_idx}
        self.next_id += 1
        return new_id

    def _update_history(self, pid, box, frame_idx):
        if pid not in self.person_history:
            self.person_history[pid] = {"bboxes": [], "last_seen": frame_idx}
        self.person_history[pid]["bboxes"].append(box.tolist())
        self.person_history[pid]["last_seen"] = frame_idx
        if len(self.person_history[pid]["bboxes"]) > 10:
            self.person_history[pid]["bboxes"].pop(0)


class TheftPredictor:

    def __init__(self, model_path, device):
        self.device    = device
        self.model     = None
        self.threshold = 0.75
        self._load(model_path)

    def _load(self, model_path):
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model not found: {model_path}")

        ckpt = torch.load(model_path, map_location=self.device)

        global HIDDEN_SIZE, NUM_LAYERS, WINDOW_SIZE
        HIDDEN_SIZE = ckpt.get("hidden_size", HIDDEN_SIZE)
        NUM_LAYERS  = ckpt.get("num_layers",  NUM_LAYERS)
        WINDOW_SIZE = ckpt.get("window_size", WINDOW_SIZE)

        self.model = TheftDetectionLSTM()
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()
        self.model.to(self.device)

        self.threshold = PRED_THRESHOLD_OVERRIDE or ckpt.get("threshold", 0.55)
        print(f"model loaded  epoch={ckpt.get('epoch','?')} | val_f1={ckpt.get('val_f1',0):.4f} | threshold={self.threshold}")

    def predict_person(self, kps_list):
        if len(kps_list) < MIN_SEQUENCE_LEN:
            return None

        windows = build_windows(kps_list)
        if len(windows) == 0:
            return None

        X = torch.tensor(windows, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            probs = torch.sigmoid(self.model(X)).squeeze(1).cpu().numpy()

        theft_count = int(np.sum(probs >= self.threshold))
        total       = len(probs)
        avg_prob    = float(np.mean(probs))
        k           = min(TOP_K_WINDOWS, total)
        topk_avg    = float(np.mean(np.sort(probs)[::-1][:k]))

        if VERDICT_MODE == "topk_avg":
            is_theft = topk_avg >= self.threshold
        elif VERDICT_MODE == "any":
            is_theft = theft_count > 0
        elif VERDICT_MODE == "majority":
            is_theft = theft_count > (total / 2)
        else:
            is_theft = avg_prob >= self.threshold

        return {
            "probs"          : probs,
            "avg_prob"       : avg_prob,
            "topk_avg"       : topk_avg,
            "theft_count"    : theft_count,
            "total_windows"  : total,
            "is_theft"       : is_theft,
        }


def normalize_keypoints(keypoints, frame_w, frame_h):
    kps        = keypoints.copy().astype(np.float32)
    kps[:, 0]  = kps[:, 0] / frame_w
    kps[:, 1]  = kps[:, 1] / frame_h
    kps[:, :2] = np.clip(kps[:, :2], 0.0, 1.0)
    return kps.flatten()


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter == 0.0:
        return 0.0
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)


def build_windows(kps_list):
    kps = np.array(kps_list, dtype=np.float32)
    T   = len(kps)

    if T < WINDOW_SIZE:
        padding = np.zeros((WINDOW_SIZE - T, INPUT_SIZE), dtype=np.float32)
        kps     = np.vstack([kps, padding])
        T       = WINDOW_SIZE

    windows = []
    for start in range(0, T - WINDOW_SIZE + 1, STRIDE):
        windows.append(kps[start : start + WINDOW_SIZE])

    last_win = kps[T - WINDOW_SIZE : T]
    if len(windows) == 0 or not np.array_equal(windows[-1], last_win):
        windows.append(last_win)

    return np.array(windows, dtype=np.float32)


def merge_tracks(raw_sequences):
    if len(raw_sequences) <= 1:
        return raw_sequences

    track_info = sorted(
        [{"id": tid, "first": 0, "last": len(frames) - 1}
         for tid, frames in raw_sequences.items()],
        key=lambda x: x["first"]
    )

    merged      = {}
    merged_into = {}

    for i, t1 in enumerate(track_info):
        if t1["id"] in merged_into:
            continue
        merged[t1["id"]] = list(raw_sequences[t1["id"]])
        for t2 in track_info[i + 1:]:
            if t2["id"] in merged_into:
                continue
            gap = t2["first"] - t1["last"]
            if 0 < gap < MERGE_TEMPORAL_GAP:
                merged[t1["id"]].extend(raw_sequences[t2["id"]])
                merged_into[t2["id"]] = t1["id"]
                t1["last"] = t2["last"]

    return {new_id: frames for new_id, (_, frames) in enumerate(sorted(merged.items()))}


def run_pipeline(video_path):
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    print(f"video: {video_path}")
    print(f"device: {DEVICE} | mode: {VERDICT_MODE} | top-k: {TOP_K_WINDOWS}")

    yolo_model = YOLO(YOLO_MODEL)

    tracker       = PersonTracker()
    raw_sequences = tracker.process_video(yolo_model, video_path)

    if not raw_sequences:
        print("no people detected")
        return {}

    final_sequences = merge_tracks(raw_sequences)
    print(f"persons after merge: {len(final_sequences)}")

    predictor = TheftPredictor(MODEL_PATH, DEVICE)
    results   = {}

    for pid, kps_list in final_sequences.items():
        res = predictor.predict_person(kps_list)
        if res is None:
            print(f"  person {pid+1}: {len(kps_list)} frames — skipped (need >={MIN_SEQUENCE_LEN})")
            continue
        res["frames_tracked"] = len(kps_list)
        res["verdict"]        = "THEFT" if res["is_theft"] else "NORMAL"
        results[pid]          = res

    if not results:
        print("no persons had enough frames")
        return {}

    any_theft = any(r["verdict"] == "THEFT" for r in results.values())

    print("\n" + "=" * 60)
    print(f"  video    : {os.path.basename(video_path)}")
    print(f"  persons  : {len(results)}")
    print(f"  threshold: {predictor.threshold} | mode: {VERDICT_MODE}")
    print("-" * 60)

    for pid, r in sorted(results.items()):
        label = "THEFT" if r["is_theft"] else "NORMAL"
        print(f"\n  person {pid+1}  ->  {label}")
        print(f"    frames tracked : {r['frames_tracked']}")
        print(f"    windows        : {r['total_windows']}  ({r['theft_count']} above threshold)")
        print(f"    avg prob       : {r['avg_prob']:.4f}")
        print(f"    top-{TOP_K_WINDOWS} avg       : {r['topk_avg']:.4f}  (verdict based on this)")

        if SHOW_PER_SEQUENCE_PROBS:
            top_indices = set(np.argsort(r["probs"])[::-1][:TOP_K_WINDOWS].tolist())
            print(f"    per-window probs:")
            for i, p in enumerate(r["probs"]):
                bar  = "|" * int(p * 20)
                flag = " <- theft" if p >= predictor.threshold else ""
                star = "*" if i in top_indices else " "
                print(f"      win {i+1:02d} {star} {p:.4f}  {bar}{flag}")

    print("\n" + "=" * 60)
    print(f"  OVERALL: {'THEFT DETECTED' if any_theft else 'NO THEFT DETECTED'}")
    print("=" * 60)

    return results


run_pipeline(VIDEO_PATH)

video: N:\Final Preparation\FULL_DATASET\new\testing\positive\theft_082.mp4
device: cpu | mode: topk_avg | top-k: 3
video: 133 frames | 2160x3840 | 30.0fps -> 24fps
  tracking done — 106 frames | 3 track(s)    
persons after merge: 3
model loaded  epoch=24 | val_f1=0.6914 | threshold=0.55
  person 2: 1 frames — skipped (need >=25)
  person 3: 5 frames — skipped (need >=25)

  video    : theft_082.mp4
  persons  : 1
  threshold: 0.55 | mode: topk_avg
------------------------------------------------------------

  person 1  ->  THEFT
    frames tracked : 106
    windows        : 9  (6 above threshold)
    avg prob       : 0.6595
    top-3 avg       : 0.9360  (verdict based on this)
    per-window probs:
      win 01   0.0732  |
      win 02   0.1250  ||
      win 03   0.4051  ||||||||
      win 04   0.8513  ||||||||||||||||| <- theft
      win 05 * 0.9056  |||||||||||||||||| <- theft
      win 06 * 0.9475  |||||||||||||||||| <- theft
      win 07 * 0.9551  ||||||||||||||||||| <- theft
  

{0: {'probs': array([   0.073178,     0.12498,     0.40509,     0.85129,     0.90557,     0.94748,      0.9551,     0.84703,     0.82554], dtype=float32),
  'avg_prob': 0.6594725847244263,
  'topk_avg': 0.9360479712486267,
  'theft_count': 6,
  'total_windows': 9,
  'is_theft': True,
  'frames_tracked': 106,
  'verdict': 'THEFT'}}